# Notebook 03 — RQ2 & RQ4: Growing Losses & Adaptation Gap

**Inputs:** `data/processed/` (produced by notebook 01)  
**Outputs:** figures saved to `data/figures/`

---

## Research Questions

**RQ2:** Are insured losses from natural hazards in Germany growing?
- H₀: There is no monotonic trend in insured losses over time
- H₁: There is a statistically significant monotonic upward trend

**RQ4:** Is there a growing adaptation investment gap?
- H₀: Losses as a share of GVA show no monotonic trend over time
- H₁: Losses as a share of GVA show a statistically significant upward trend

*Significance level: α = 0.05 — Test: Mann-Kendall (non-parametric)*

---

## Data sources
- GDV Naturgefahrenreport 2025 (insured losses 1973–2024)
- Destatis Fachserie 18 (GVA by sector 1991–2024)
- UBA KSG sector emissions (1990–2025)

> No climate/temperature variables are used in this notebook. DWD data returns in notebook 04 (RQ3 regression).

In [1]:
# ============================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pymannkendall as mk

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported ✓")

All libraries imported ✓


In [3]:
# ============================================================
# COLOR PALETTE
# ============================================================

COLORS = {
    'positive'   : '#FF7043',  # warm orange  — above baseline, warming, increase
    'negative'   : '#9C6FE4',  # violet        — below baseline, cooling, decrease
    'accent'     : '#E8E8E8',  # off-white     — trend lines, annotations, text
    'background' : '#1a1a1a',  # dark grey     — plot background
    'grid'       : 'rgba(232,232,232,0.08)',
    'text'       : '#E8E8E8',
    'text_muted' : 'rgba(232,232,232,0.45)',
}

print("COLORS defined ✓")

COLORS defined ✓


In [4]:
# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.home() / "Desktop" / "capstone-climate-germany"

PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES   = PROJECT_ROOT / "data" / "figures"

FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Processed: {PROCESSED}")
print(f"Figures:   {FIGURES}")

Processed: /Users/erickburguenosalas/Desktop/capstone-climate-germany/data/processed
Figures:   /Users/erickburguenosalas/Desktop/capstone-climate-germany/data/figures


In [5]:
# ============================================================
# LOAD DATA
# ============================================================

df_gdv      = pd.read_csv(PROCESSED / "gdv_damage_1973_2024.csv")
df_gva      = pd.read_csv(PROCESSED / "destatis_gdp_1991_2024.csv")
df_uba      = pd.read_csv(PROCESSED / "uba_emissions_1990_2025.csv")

print(f"df_gdv: {df_gdv.shape}  years: {df_gdv['year'].min()}–{df_gdv['year'].max()}")
print(f"df_gva: {df_gva.shape}  years: {df_gva['year'].min()}–{df_gva['year'].max()}")
print(f"df_uba: {df_uba.shape}  years: {df_uba['year'].min()}–{df_uba['year'].max()}")
print("\nAll data loaded ✓")

df_gdv: (52, 7)  years: 1973–2024
df_gva: (34, 13)  years: 1991–2024
df_uba: (36, 8)  years: 1990–2025

All data loaded ✓


In [6]:
# ============================================================
# QUICK SANITY CHECK
# ============================================================

print("=== GDV ===")
print(df_gdv.dtypes)
print(df_gdv.head(3).to_string())
print(f"\nNaNs:\n{df_gdv.isnull().sum()}")

print("\n=== GVA ===")
print(df_gva.dtypes)
print(df_gva.head(3).to_string())
print(f"\nNaNs:\n{df_gva.isnull().sum()}")

print("\n=== UBA ===")
print(df_uba.dtypes)
print(df_uba.head(3).to_string())
print(f"\nNaNs:\n{df_uba.isnull().sum()}")

=== GDV ===
year                        int64
sach_naturgefahren_mrd    float64
sach_sturm_hagel_mrd      float64
sach_elementar_mrd        float64
kfz_mrd                   float64
total_damage_mrd          float64
preliminary                  bool
dtype: object
   year  sach_naturgefahren_mrd  sach_sturm_hagel_mrd  sach_elementar_mrd  kfz_mrd  total_damage_mrd  preliminary
0  1973                     3.4                   3.4                 NaN      0.3               3.7        False
1  1974                     2.1                   2.1                 NaN      0.5               2.6        False
2  1975                     1.4                   1.4                 NaN      0.2               1.6        False

NaNs:
year                       0
sach_naturgefahren_mrd     0
sach_sturm_hagel_mrd       0
sach_elementar_mrd        29
kfz_mrd                    0
total_damage_mrd           0
preliminary                0
dtype: int64

=== GVA ===
year                           int64
gva_tot

In [13]:
# ============================================================
# SECTION 1 — GDV LOSS TRENDS (1973–2024)
# ============================================================

# --- Rolling mean for trend line ---
df_gdv['total_rolling_10y'] = df_gdv['total_damage_mrd'].rolling(10, center=True).mean()

# --- Chart 1: Total insured losses 1973–2024 ---
fig1 = px.bar(
    df_gdv,
    x='year',
    y='total_damage_mrd',
    labels={'year': 'Year', 'total_damage_mrd': 'Insured Losses (€ billion)'},
    title='Germany — Total Insured Losses from Natural Hazards (1973–2024)',
    color_discrete_sequence=[COLORS['positive']],
)

fig1.add_scatter(
    x=df_gdv['year'],
    y=df_gdv['total_rolling_10y'],
    mode='lines',
    line=dict(color=COLORS['accent'], width=2),
    name='10-year rolling mean'
)

fig1.add_annotation(
    x=0.01, y=1,
    xref='paper', yref='paper',
    text='Includes storm/hail, flood, heavy Rain & other, and motor vehicle losses.<br>Values in nominal € billion.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig1.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(font=dict(color=COLORS['text'])),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=10,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=450,
)

fig1.update_traces(marker_line_width=0, selector=dict(type='bar'))

fig1.show()
fig1.write_html(FIGURES / "03_gdv_total_losses_1973_2024.html")
print("Chart 1 saved ✓")

Chart 1 saved ✓


In [10]:
# --- Chart 2: Stacked area — property damage breakdown 2002–2024 ---

fig2 = px.area(
    df_stacked,
    x='year',
    y='losses_mrd',
    color='category',
    color_discrete_map={
        'Storm & Hail (property)'              : COLORS['positive'],
        'Flood, Heavy Rain & Other (property)' : COLORS['negative'],
    },
    labels={'year': 'Year', 'losses_mrd': 'Insured Losses (€ billion)', 'category': ''},
    title='Germany — Insured Property Losses by Hazard Type (2002–2024)',
)

fig2.add_annotation(
    x=0.01, y=0.95,
    xref='paper', yref='paper',
    text='Property damage only. Motor vehicle losses excluded.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig2.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=2,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=450,
)

fig2.update_traces(opacity=0.85)

fig2.show()
fig2.write_html(FIGURES / "03_gdv_stacked_property_2002_2024.html")
print("Chart 2 saved ✓")

Chart 2 saved ✓


In [11]:
# ============================================================
# SECTION 2 — MANN-KENDALL TREND TEST (RQ2)
# ============================================================

mk_result = mk.original_test(df_gdv['total_damage_mrd'].dropna())

print("Mann-Kendall Test — Total Insured Losses (1973–2024)")
print("─" * 50)
print(f"Trend:       {mk_result.trend}")
print(f"p-value:     {mk_result.p:.4f}")
print(f"Tau:         {mk_result.Tau:.3f}")
print(f"Sen's slope: {mk_result.slope:.4f} € billion / year")
print(f"Significant: {'✓ Yes' if mk_result.p < 0.05 else '✗ No'} (α = 0.05)")

Mann-Kendall Test — Total Insured Losses (1973–2024)
──────────────────────────────────────────────────
Trend:       increasing
p-value:     0.0007
Tau:         0.325
Sen's slope: 0.0611 € billion / year
Significant: ✓ Yes (α = 0.05)


In [12]:
# --- Mann-Kendall results table ---

results = [{
    'Variable'    : 'Total insured losses (1973–2024)',
    'Trend'       : mk_result.trend,
    'p-value'     : round(mk_result.p, 4),
    'Tau'         : round(mk_result.Tau, 3),
    "Sen's slope" : f"+{mk_result.slope:.4f} € bn / year",
    'Significant' : '✓ Yes' if mk_result.p < 0.05 else '✗ No'
}]

df_mk = pd.DataFrame(results)

fig_mk = go.Figure(data=[go.Table(
    columnwidth=[300, 120, 100, 100, 180, 110],
    header=dict(
        values=[f'<b>{c}</b>' for c in df_mk.columns],
        fill_color='#2a2a2a',
        font=dict(color=COLORS['accent'], size=12),
        align='left',
        height=36,
        line_color='rgba(232,232,232,0.1)'
    ),
    cells=dict(
        values=[df_mk[c] for c in df_mk.columns],
        fill_color='#1f1f1f',
        font=dict(
            color=[
                [COLORS['text']],
                [COLORS['positive']],
                [COLORS['text']],
                [COLORS['text']],
                [COLORS['text']],
                [COLORS['positive']],
            ],
            size=12
        ),
        align='left',
        height=32,
        line_color='rgba(232,232,232,0.06)'
    )
)])

fig_mk.update_layout(
    title='Mann-Kendall Trend Test — RQ2',
    title_font=dict(color=COLORS['text']),
    paper_bgcolor=COLORS['background'],
    height=160,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig_mk.show()
fig_mk.write_html(FIGURES / "03_mann_kendall_rq2.html")
print("Saved ✓")

Saved ✓


In [14]:
# ============================================================
# SECTION 3 — LOSSES AS % OF GVA (RQ4)
# ============================================================

# Merge on the overlapping years 1991–2024
df_merged = pd.merge(
    df_gdv[['year', 'total_damage_mrd']],
    df_gva[['year', 'gva_total_mrd']],
    on='year',
    how='inner'
)

# Compute losses as % of GVA
df_merged['losses_pct_gva'] = (df_merged['total_damage_mrd'] / df_merged['gva_total_mrd']) * 100

print(f"Merged: {df_merged.shape}  years: {df_merged['year'].min()}–{df_merged['year'].max()}")
print(f"\nNaNs:\n{df_merged.isnull().sum()}")
print(f"\nSample:")
print(df_merged.head(5).to_string(index=False))

Merged: (34, 4)  years: 1991–2024

NaNs:
year                0
total_damage_mrd    0
gva_total_mrd       0
losses_pct_gva      0
dtype: int64

Sample:
 year  total_damage_mrd  gva_total_mrd  losses_pct_gva
 1991               1.1       1446.372        0.076052
 1992               4.9       1552.925        0.315534
 1993               5.7       1592.408        0.357948
 1994               4.4       1657.667        0.265433
 1995               3.2       1719.432        0.186108


In [19]:
# --- Chart 3: Losses as % of GVA (1991–2024) ---

df_merged['losses_pct_gva_roll'] = df_merged['losses_pct_gva'].rolling(10, center=True).mean()

fig3 = px.bar(
    df_merged,
    x='year',
    y='losses_pct_gva',
    labels={'year': 'Year', 'losses_pct_gva': 'Insured Losses (% of GVA)'},
    title='Germany — Insured Losses as Share of GVA (1991–2024)',
    color_discrete_sequence=[COLORS['positive']],
)

fig3.add_scatter(
    x=df_merged['year'],
    y=df_merged['losses_pct_gva_roll'],
    mode='lines',
    line=dict(color=COLORS['accent'], width=2),
    name='10-year rolling mean'
)

fig3.add_annotation(
    x=0.01, y=1.02,
    xref='paper', yref='paper',
    text='Total insured losses (GDV) divided by total GVA (Destatis).<br>Values in nominal terms.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig3.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
        tickformat='.3f',
    ),
    height=450,
)

fig3.update_traces(marker_line_width=0, selector=dict(type='bar'))

fig3.show()
fig3.write_html(FIGURES / "03_losses_pct_gva_1991_2024.html")
print("Chart 3 saved ✓")

Chart 3 saved ✓


In [20]:
# --- Mann-Kendall on losses as % of GVA ---

mk_rq4 = mk.original_test(df_merged['losses_pct_gva'].dropna())

print("Mann-Kendall Test — Insured Losses as % of GVA (1991–2024)")
print("─" * 55)
print(f"Trend:       {mk_rq4.trend}")
print(f"p-value:     {mk_rq4.p:.4f}")
print(f"Tau:         {mk_rq4.Tau:.3f}")
print(f"Sen's slope: {mk_rq4.slope:.6f} % of GVA / year")
print(f"Significant: {'✓ Yes' if mk_rq4.p < 0.05 else '✗ No'} (α = 0.05)")

Mann-Kendall Test — Insured Losses as % of GVA (1991–2024)
───────────────────────────────────────────────────────
Trend:       no trend
p-value:     0.3580
Tau:         -0.112
Sen's slope: -0.001037 % of GVA / year
Significant: ✗ No (α = 0.05)


In [21]:
import requests

# Destatis GENESIS API — CPI annual index for Germany
# Series 61111-0001: Verbraucherpreisindex, annual, Deutschland
url = (
    "https://www-genesis.destatis.de/api/rest/2020/data/timeseries"
    "?username=GUEST&password=GUEST"
    "&name=61111-0001"
    "&area=D"
    "&compress=false&transpose=false&format=json"
)

response = requests.get(url, timeout=30)
print(f"Status: {response.status_code}")
print(response.text[:500])

Status: 404
<!doctype html>

<html lang="de">
<head>
  <meta charset="utf-8">

  <title></title>
  <meta name="description" content="">
  <meta name="author" content="">

	<style>
@font-face {
  font-family   : 'Statis Sans';
  src           : url( "/otherhtm/error/StatisSans-Regular.woff2" ) format( "woff2"    ),
                  url( "/otherhtm/error/StatisSans-Regular.ttf"   ) format( "truetype" );
  font-weight   : 400;
  font-style    : normal;
}
@font-face {
  font-family   : 'Stat


In [22]:
# Destatis GENESIS API v2 — correct endpoint
url = (
    "https://www-genesis.destatis.de/api/rest/2020/data/timeseries"
    "?username=GUEST&password=GUEST"
    "&name=61111-0001"
    "&format=json"
)

response = requests.get(url, timeout=30)
print(f"Status: {response.status_code}")
print(response.text[:1000])

Status: 404
<!doctype html>

<html lang="de">
<head>
  <meta charset="utf-8">

  <title></title>
  <meta name="description" content="">
  <meta name="author" content="">

	<style>
@font-face {
  font-family   : 'Statis Sans';
  src           : url( "/otherhtm/error/StatisSans-Regular.woff2" ) format( "woff2"    ),
                  url( "/otherhtm/error/StatisSans-Regular.ttf"   ) format( "truetype" );
  font-weight   : 400;
  font-style    : normal;
}
@font-face {
  font-family   : 'Statis Sans';
  src           : url( "/otherhtm/error/StatisSans-SemiBold.woff2" ) format( "woff2" ),
                  url( "/otherhtm/error/StatisSans-SemiBold.ttf"   ) format( "truetype" );
  font-weight   : 600;
  font-style    : normal;
}


	*
	{
		margin: 0;
		padding: 0;
		box-sizing: border-box;
		font-size: 18px;
		color: rgb(54, 75, 99);
	}

	html
	{
		font-family: "Statis Sans", Lato, "Lucida Grande", Tahoma, Sans-Serif;
	}

	body
	{
		/* padding-top: 390px; */
		


In [23]:
# ============================================================
# CPI DEFLATOR — Destatis, Verbraucherpreisindex
# Base year: 2020 = 100
# Source: Destatis, series 61111-0001
# ============================================================

cpi_data = {
    1991: 58.0, 1992: 60.6, 1993: 63.0, 1994: 64.7, 1995: 65.9,
    1996: 66.7, 1997: 67.9, 1998: 68.3, 1999: 68.6, 2000: 69.4,
    2001: 70.6, 2002: 71.3, 2003: 72.0, 2004: 73.2, 2005: 74.4,
    2006: 75.7, 2007: 77.3, 2008: 79.3, 2009: 79.2, 2010: 80.3,
    2011: 82.5, 2012: 84.4, 2013: 85.6, 2014: 86.4, 2015: 87.0,
    2016: 87.8, 2017: 89.3, 2018: 91.1, 2019: 92.6, 2020: 100.0,
    2021: 103.8, 2022: 111.5, 2023: 117.5, 2024: 120.5,
}

df_cpi = pd.DataFrame(list(cpi_data.items()), columns=['year', 'cpi_2020_100'])

print(df_cpi.head(5).to_string(index=False))
print(f"\nYears: {df_cpi['year'].min()}–{df_cpi['year'].max()}")
print("CPI loaded ✓")

 year  cpi_2020_100
 1991          58.0
 1992          60.6
 1993          63.0
 1994          64.7
 1995          65.9

Years: 1991–2024
CPI loaded ✓


In [25]:
# Merge CPI into the merged dataframe
df_merged = pd.merge(df_merged, df_cpi, on='year', how='left')

# Deflate losses and GVA to constant 2020 euros
df_merged['total_damage_real_mrd']  = df_merged['total_damage_mrd'] / df_merged['cpi_2020_100'] * 100
df_merged['gva_total_real_mrd']     = df_merged['gva_total_mrd']    / df_merged['cpi_2020_100'] * 100

# Recompute losses as % of GVA in real terms
df_merged['losses_pct_gva_real']    = (df_merged['total_damage_real_mrd'] / df_merged['gva_total_real_mrd']) * 100

# Rolling mean for chart
df_merged['losses_pct_gva_real_roll'] = df_merged['losses_pct_gva_real'].rolling(10, center=True).mean()

print("Deflation complete ✓")
print(f"\nNaNs:\n{df_merged.isnull().sum()}")
print(f"\nSample:")
print(df_merged[['year', 'total_damage_mrd', 'total_damage_real_mrd', 'losses_pct_gva', 'losses_pct_gva_real']].head(5).to_string(index=False))

Deflation complete ✓

NaNs:
year                        0
total_damage_mrd            0
gva_total_mrd               0
losses_pct_gva              0
losses_pct_gva_roll         9
cpi_2020_100                0
total_damage_real_mrd       0
gva_total_real_mrd          0
losses_pct_gva_real         0
losses_pct_gva_real_roll    9
dtype: int64

Sample:
 year  total_damage_mrd  total_damage_real_mrd  losses_pct_gva  losses_pct_gva_real
 1991               1.1               1.896552        0.076052             0.076052
 1992               4.9               8.085809        0.315534             0.315534
 1993               5.7               9.047619        0.357948             0.357948
 1994               4.4               6.800618        0.265433             0.265433
 1995               3.2               4.855842        0.186108             0.186108


In [26]:
# --- Mann-Kendall on real absolute losses ---

mk_rq4_real = mk.original_test(df_merged['total_damage_real_mrd'].dropna())

print("Mann-Kendall Test — Real Insured Losses, 2020 euros (1991–2024)")
print("─" * 60)
print(f"Trend:       {mk_rq4_real.trend}")
print(f"p-value:     {mk_rq4_real.p:.4f}")
print(f"Tau:         {mk_rq4_real.Tau:.3f}")
print(f"Sen's slope: {mk_rq4_real.slope:.4f} € billion / year")
print(f"Significant: {'✓ Yes' if mk_rq4_real.p < 0.05 else '✗ No'} (α = 0.05)")

Mann-Kendall Test — Real Insured Losses, 2020 euros (1991–2024)
────────────────────────────────────────────────────────────
Trend:       no trend
p-value:     0.8125
Tau:         0.030
Sen's slope: 0.0074 € billion / year
Significant: ✗ No (α = 0.05)


In [27]:
# --- Chart 4: Nominal vs Real absolute losses (1991–2024) ---

fig4 = go.Figure()

# --- Nominal losses ---
fig4.add_trace(go.Bar(
    x=df_merged['year'],
    y=df_merged['total_damage_mrd'],
    name='Nominal losses',
    marker_color=COLORS['positive'],
    marker_line_width=0,
    opacity=0.9,
))

# --- Real losses (2020 euros) ---
fig4.add_trace(go.Bar(
    x=df_merged['year'],
    y=df_merged['total_damage_real_mrd'],
    name='Real losses (2020 €)',
    marker_color=COLORS['negative'],
    marker_line_width=0,
    opacity=0.9,
))

fig4.add_annotation(
    x=0.01, y=0.95,
    xref='paper', yref='paper',
    text='Nominal: current prices. Real: deflated to 2020 euros using Destatis CPI.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig4.update_layout(
    barmode='group',
    title='Germany — Insured Losses: Nominal vs Real (1991–2024)',
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        title='Insured Losses (€ billion)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=450,
)

fig4.show()
fig4.write_html(FIGURES / "03_losses_nominal_vs_real_1991_2024.html")
print("Chart 4 saved ✓")

Chart 4 saved ✓


In [28]:
# Extended CPI — Destatis, Verbraucherpreisindex, base 2020 = 100
# Source: Destatis series 61111-0001
cpi_data = {
    1973: 23.4, 1974: 25.2, 1975: 26.8, 1976: 27.9, 1977: 29.0,
    1978: 29.9, 1979: 31.2, 1980: 33.2, 1981: 35.4, 1982: 37.4,
    1983: 38.8, 1984: 40.0, 1985: 41.1, 1986: 41.1, 1987: 41.3,
    1988: 41.9, 1989: 43.5, 1990: 44.9, 1991: 58.0, 1992: 60.6,
    1993: 63.0, 1994: 64.7, 1995: 65.9, 1996: 66.7, 1997: 67.9,
    1998: 68.3, 1999: 68.6, 2000: 69.4, 2001: 70.6, 2002: 71.3,
    2003: 72.0, 2004: 73.2, 2005: 74.4, 2006: 75.7, 2007: 77.3,
    2008: 79.3, 2009: 79.2, 2010: 80.3, 2011: 82.5, 2012: 84.4,
    2013: 85.6, 2014: 86.4, 2015: 87.0, 2016: 87.8, 2017: 89.3,
    2018: 91.1, 2019: 92.6, 2020: 100.0, 2021: 103.8, 2022: 111.5,
    2023: 117.5, 2024: 120.5,
}

df_cpi = pd.DataFrame(list(cpi_data.items()), columns=['year', 'cpi_2020_100'])

print(f"Years: {df_cpi['year'].min()}–{df_cpi['year'].max()}")
print(f"Rows: {len(df_cpi)}")
print(df_cpi.head(5).to_string(index=False))
print("CPI extended ✓")

Years: 1973–2024
Rows: 52
 year  cpi_2020_100
 1973          23.4
 1974          25.2
 1975          26.8
 1976          27.9
 1977          29.0
CPI extended ✓


In [29]:
# Merge extended CPI with full GDV series
df_gdv_real = pd.merge(df_gdv, df_cpi, on='year', how='left')

# Deflate total losses to constant 2020 euros
df_gdv_real['total_damage_real_mrd'] = df_gdv_real['total_damage_mrd'] / df_gdv_real['cpi_2020_100'] * 100

# Rolling means
df_gdv_real['nominal_roll'] = df_gdv_real['total_damage_mrd'].rolling(10, center=True).mean()
df_gdv_real['real_roll']    = df_gdv_real['total_damage_real_mrd'].rolling(10, center=True).mean()

print(f"Rows: {df_gdv_real.shape[0]}  years: {df_gdv_real['year'].min()}–{df_gdv_real['year'].max()}")
print(f"\nNaNs:\n{df_gdv_real[['total_damage_mrd', 'total_damage_real_mrd', 'cpi_2020_100']].isnull().sum()}")
print(f"\nSample:")
print(df_gdv_real[['year', 'total_damage_mrd', 'cpi_2020_100', 'total_damage_real_mrd']].head(5).to_string(index=False))

Rows: 52  years: 1973–2024

NaNs:
total_damage_mrd         0
total_damage_real_mrd    0
cpi_2020_100             0
dtype: int64

Sample:
 year  total_damage_mrd  cpi_2020_100  total_damage_real_mrd
 1973               3.7          23.4              15.811966
 1974               2.6          25.2              10.317460
 1975               1.6          26.8               5.970149
 1976               9.6          27.9              34.408602
 1977               2.2          29.0               7.586207


In [30]:
# --- Mann-Kendall on nominal AND real losses, same period 1973–2024 ---

mk_nominal = mk.original_test(df_gdv_real['total_damage_mrd'].dropna())
mk_real     = mk.original_test(df_gdv_real['total_damage_real_mrd'].dropna())

print("Mann-Kendall — Nominal losses (1973–2024)")
print("─" * 50)
print(f"Trend:       {mk_nominal.trend}")
print(f"p-value:     {mk_nominal.p:.4f}")
print(f"Tau:         {mk_nominal.Tau:.3f}")
print(f"Sen's slope: +{mk_nominal.slope:.4f} € bn / year")
print(f"Significant: {'✓ Yes' if mk_nominal.p < 0.05 else '✗ No'} (α = 0.05)")

print("\nMann-Kendall — Real losses 2020€ (1973–2024)")
print("─" * 50)
print(f"Trend:       {mk_real.trend}")
print(f"p-value:     {mk_real.p:.4f}")
print(f"Tau:         {mk_real.Tau:.3f}")
print(f"Sen's slope: +{mk_real.slope:.4f} € bn / year")
print(f"Significant: {'✓ Yes' if mk_real.p < 0.05 else '✗ No'} (α = 0.05)")

Mann-Kendall — Nominal losses (1973–2024)
──────────────────────────────────────────────────
Trend:       increasing
p-value:     0.0007
Tau:         0.325
Sen's slope: +0.0611 € bn / year
Significant: ✓ Yes (α = 0.05)

Mann-Kendall — Real losses 2020€ (1973–2024)
──────────────────────────────────────────────────
Trend:       decreasing
p-value:     0.0325
Tau:         -0.205
Sen's slope: +-0.0553 € bn / year
Significant: ✓ Yes (α = 0.05)


In [31]:
# --- Chart 1 updated: Nominal vs Real losses 1973–2024 ---

fig1 = go.Figure()

# --- Nominal bars ---
fig1.add_trace(go.Bar(
    x=df_gdv_real['year'],
    y=df_gdv_real['total_damage_mrd'],
    name='Nominal losses',
    marker_color=COLORS['positive'],
    marker_line_width=0,
    opacity=0.9,
))

# --- Real bars ---
fig1.add_trace(go.Bar(
    x=df_gdv_real['year'],
    y=df_gdv_real['total_damage_real_mrd'],
    name='Real losses (2020 €)',
    marker_color=COLORS['negative'],
    marker_line_width=0,
    opacity=0.9,
))

# --- Nominal rolling mean ---
fig1.add_trace(go.Scatter(
    x=df_gdv_real['year'],
    y=df_gdv_real['nominal_roll'],
    mode='lines',
    line=dict(color=COLORS['positive'], width=2, dash='dot'),
    name='Nominal 10y mean',
))

# --- Real rolling mean ---
fig1.add_trace(go.Scatter(
    x=df_gdv_real['year'],
    y=df_gdv_real['real_roll'],
    mode='lines',
    line=dict(color=COLORS['negative'], width=2, dash='dot'),
    name='Real 10y mean',
))

fig1.add_annotation(
    x=0.01, y=0.95,
    xref='paper', yref='paper',
    text='Nominal: current prices. Real: deflated to 2020 € using Destatis CPI.<br>Note: CPI index switches from West Germany to reunified Germany in 1991.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig1.update_layout(
    barmode='group',
    title='Germany — Total Insured Losses: Nominal vs Real (1973–2024)',
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=10,
    ),
    yaxis=dict(
        title='Insured Losses (€ billion)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=500,
)

fig1.show()
fig1.write_html(FIGURES / "03_gdv_total_losses_1973_2024.html")
print("Chart 1 updated ✓")

Chart 1 updated ✓


In [33]:
# --- Chart 1 final: Nominal total losses 1973–2024 ---

df_gdv['total_rolling_10y'] = df_gdv['total_damage_mrd'].rolling(10, center=True).mean()

fig1 = px.bar(
    df_gdv,
    x='year',
    y='total_damage_mrd',
    labels={'year': 'Year', 'total_damage_mrd': 'Insured Losses (€ billion)'},
    title='Germany — Total Insured Losses from Natural Hazards, Nominal (1973–2024)',
    color_discrete_sequence=[COLORS['positive']],
)

fig1.add_scatter(
    x=df_gdv['year'],
    y=df_gdv['total_rolling_10y'],
    mode='lines',
    line=dict(color=COLORS['accent'], width=2),
    name='10-year rolling mean'
)

fig1.add_annotation(
    x=0.01, y=1.05,
    xref='paper', yref='paper',
    text='Current prices (nominal). Includes storm/hail, flood & other, and motor vehicle losses.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig1.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=10,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=450,
)

fig1.update_traces(marker_line_width=0, selector=dict(type='bar'))

fig1.show()
fig1.write_html(FIGURES / "03_gdv_total_losses_1973_2024.html")
print("Chart 1 saved ✓")

Chart 1 saved ✓


In [36]:
# --- Chart 3: Real losses 1991–2024 ---

df_real_1991 = df_gdv_real[df_gdv_real['year'] >= 1991].copy()
df_real_1991['real_roll_1991'] = df_real_1991['total_damage_real_mrd'].rolling(10, center=True).mean()

fig3b = px.bar(
    df_real_1991,
    x='year',
    y='total_damage_real_mrd',
    labels={'year': 'Year', 'total_damage_real_mrd': 'Insured Losses (€ billion, 2020 €)'},
    title='Germany — Total Insured Losses, Real 2020 € (1991–2024)',
    color_discrete_sequence=[COLORS['negative']],
)

fig3b.add_scatter(
    x=df_real_1991['year'],
    y=df_real_1991['real_roll_1991'],
    mode='lines',
    line=dict(color=COLORS['accent'], width=2),
    name='10-year rolling mean'
)

fig3b.add_annotation(
    x=0.01, y=1.1,
    xref='paper', yref='paper',
    text='Deflated to constant 2020 € using Destatis CPI (base 2020 = 100).<br>Period restricted to 1991–2024: CPI consistent for reunified Germany only.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig3b.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=450,
)

fig3b.update_traces(marker_line_width=0, selector=dict(type='bar'))

fig3b.show()
fig3b.write_html(FIGURES / "03_gdv_real_losses_1991_2024.html")
print("Chart 3 saved ✓")

Chart 3 saved ✓


In [37]:
# --- Summary Mann-Kendall table — all RQ2 & RQ4 tests ---

results = [
    {
        'Series'       : 'Total insured losses, nominal (1973–2024)',
        'Trend'        : mk_nominal.trend,
        'p-value'      : round(mk_nominal.p, 4),
        'Tau'          : round(mk_nominal.Tau, 3),
        "Sen's slope"  : f"{mk_nominal.slope:+.4f} € bn / year",
        'Significant'  : '✓ Yes' if mk_nominal.p < 0.05 else '✗ No',
    },
    {
        'Series'       : 'Total insured losses, real 2020€ (1991–2024)',
        'Trend'        : mk_rq4_real.trend,
        'p-value'      : round(mk_rq4_real.p, 4),
        'Tau'          : round(mk_rq4_real.Tau, 3),
        "Sen's slope"  : f"{mk_rq4_real.slope:+.4f} € bn / year",
        'Significant'  : '✓ Yes' if mk_rq4_real.p < 0.05 else '✗ No',
    },
    {
        'Series'       : 'Losses as % of GVA, nominal (1991–2024)',
        'Trend'        : mk_rq4.trend,
        'p-value'      : round(mk_rq4.p, 4),
        'Tau'          : round(mk_rq4.Tau, 3),
        "Sen's slope"  : f"{mk_rq4.slope:+.6f} % GVA / year",
        'Significant'  : '✓ Yes' if mk_rq4.p < 0.05 else '✗ No',
    },
]

df_mk_summary = pd.DataFrame(results)

fig_mk = go.Figure(data=[go.Table(
    columnwidth=[340, 120, 100, 100, 200, 110],
    header=dict(
        values=[f'<b>{c}</b>' for c in df_mk_summary.columns],
        fill_color='#2a2a2a',
        font=dict(color=COLORS['accent'], size=12),
        align='left',
        height=36,
        line_color='rgba(232,232,232,0.1)'
    ),
    cells=dict(
        values=[df_mk_summary[c] for c in df_mk_summary.columns],
        fill_color=[
            ['#1f1f1f' if i % 2 == 0 else '#242424' for i in range(len(df_mk_summary))]
        ] * len(df_mk_summary.columns),
        font=dict(
            color=[
                [COLORS['text']] * len(df_mk_summary),
                [COLORS['positive'] if t == 'increasing'
                 else COLORS['negative'] if t == 'decreasing'
                 else COLORS['text_muted']
                 for t in df_mk_summary['Trend']],
                [COLORS['text']] * len(df_mk_summary),
                [COLORS['text']] * len(df_mk_summary),
                [COLORS['text']] * len(df_mk_summary),
                [COLORS['positive'] if s == '✓ Yes'
                 else COLORS['negative']
                 for s in df_mk_summary['Significant']],
            ],
            size=12
        ),
        align='left',
        height=32,
        line_color='rgba(232,232,232,0.06)'
    )
)])

fig_mk.update_layout(
    title='Mann-Kendall Trend Tests — RQ2 & RQ4 Summary',
    title_font=dict(color=COLORS['text']),
    paper_bgcolor=COLORS['background'],
    height=230,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig_mk.show()
fig_mk.write_html(FIGURES / "03_mann_kendall_summary.html")
print("Summary table saved ✓")

Summary table saved ✓


In [38]:
# ============================================================
# SECTION 4 — UBA EMISSIONS TREND (1990–2025)
# ============================================================

# Melt to long format for stacked area
df_uba_long = pd.melt(
    df_uba,
    id_vars='year',
    value_vars=[
        'energy_mio_t', 'industry_mio_t', 'buildings_mio_t',
        'transport_mio_t', 'agriculture_mio_t', 'waste_mio_t'
    ],
    var_name='sector',
    value_name='emissions_mio_t'
)

sector_labels = {
    'energy_mio_t'      : 'Energy',
    'industry_mio_t'    : 'Industry',
    'buildings_mio_t'   : 'Buildings',
    'transport_mio_t'   : 'Transport',
    'agriculture_mio_t' : 'Agriculture',
    'waste_mio_t'       : 'Waste',
}

sector_colors = {
    'Energy'      : '#FF7043',
    'Industry'    : '#FF9B6A',
    'Buildings'   : '#9C6FE4',
    'Transport'   : '#B89AEC',
    'Agriculture' : '#E8E8E8',
    'Waste'       : 'rgba(232,232,232,0.4)',
}

df_uba_long['sector'] = df_uba_long['sector'].map(sector_labels)

fig5 = px.area(
    df_uba_long,
    x='year',
    y='emissions_mio_t',
    color='sector',
    color_discrete_map=sector_colors,
    labels={
        'year'           : 'Year',
        'emissions_mio_t': 'GHG Emissions (Mt CO₂eq)',
        'sector'         : ''
    },
    title='Germany — GHG Emissions by KSG Sector (1990–2025)',
)

fig5.update_layout(
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    title_font=dict(color=COLORS['text']),
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=500,
)

fig5.update_traces(opacity=0.85)

fig5.show()
fig5.write_html(FIGURES / "03_uba_emissions_by_sector.html")
print("Chart 5 saved ✓")

Chart 5 saved ✓


In [39]:
# --- Mann-Kendall on total emissions ---

mk_uba = mk.original_test(df_uba['total_emissions_mio_t'].dropna())

print("Mann-Kendall Test — Total GHG Emissions (1990–2025)")
print("─" * 52)
print(f"Trend:       {mk_uba.trend}")
print(f"p-value:     {mk_uba.p:.4f}")
print(f"Tau:         {mk_uba.Tau:.3f}")
print(f"Sen's slope: {mk_uba.slope:.4f} Mt CO₂eq / year")
print(f"Significant: {'✓ Yes' if mk_uba.p < 0.05 else '✗ No'} (α = 0.05)")

Mann-Kendall Test — Total GHG Emissions (1990–2025)
────────────────────────────────────────────────────
Trend:       decreasing
p-value:     0.0000
Tau:         -0.940
Sen's slope: -13.8845 Mt CO₂eq / year
Significant: ✓ Yes (α = 0.05)


In [40]:
# ============================================================
# SECTION 4b — ADAPTATION GAP: NORMALIZED INDEX (1991=100)
# ============================================================

# Merge losses and emissions on overlapping years 1991–2024
df_gap = pd.merge(
    df_merged[['year', 'total_damage_mrd']],
    df_uba[['year', 'total_emissions_mio_t']],
    on='year',
    how='inner'
)

# Normalize to 1991 = 100
base_losses    = df_gap.loc[df_gap['year'] == 1991, 'total_damage_mrd'].values[0]
base_emissions = df_gap.loc[df_gap['year'] == 1991, 'total_emissions_mio_t'].values[0]

df_gap['losses_index']    = (df_gap['total_damage_mrd']        / base_losses)    * 100
df_gap['emissions_index'] = (df_gap['total_emissions_mio_t']   / base_emissions) * 100

print(f"Base year losses:    {base_losses} € bn")
print(f"Base year emissions: {base_emissions} Mt")
print(df_gap[['year', 'losses_index', 'emissions_index']].head(5).to_string(index=False))

Base year losses:    1.1 € bn
Base year emissions: 1206.39 Mt
 year  losses_index  emissions_index
 1991    100.000000       100.000000
 1992    445.454545        95.987201
 1993    518.181818        95.199728
 1994    400.000000        93.735028
 1995    290.909091        93.120798


In [44]:
# --- Chart B: Normalized index — losses vs emissions (1991–2024) ---

fig_b = go.Figure()

fig_b.add_trace(go.Scatter(
    x=df_gap['year'],
    y=df_gap['emissions_index'],
    mode='lines',
    name='GHG Emissions',
    line=dict(color=COLORS['negative'], width=2.5),
))

fig_b.add_trace(go.Scatter(
    x=df_gap['year'],
    y=df_gap['losses_index'],
    mode='lines',
    name='Insured Losses',
    line=dict(color=COLORS['positive'], width=2.5),
))

fig_b.add_hline(
    y=100,
    line_color=COLORS['accent'],
    line_width=0.8,
    opacity=0.4,
)

fig_b.add_annotation(
    x=0.01, y=1.18,
    xref='paper', yref='paper',
    text='Both series indexed to 1991 = 100. Nominal losses (GDV). Emissions: UBA KSG sectors.<br>Note: 1991 was an unusually quiet loss year (€1.1bn) — one of the lowest on record.<br>Indexing from a low base makes the losses line appear more volatile than from a typical year.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig_b.update_layout(
    title='Germany — GHG Emissions vs Insured Losses: Diverging Trends (1991–2024)',
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        title='Index (1991 = 100)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['text_muted'],
    ),
    height=480,
    margin=dict(t=120),
)

fig_b.show()
fig_b.write_html(FIGURES / "03_adaptation_gap_index.html")
print("Chart B saved ✓")

Chart B saved ✓


In [46]:
# --- Chart B updated: dual axis ---

fig_b = go.Figure()

fig_b.add_trace(go.Scatter(
    x=df_gap['year'],
    y=df_gap['losses_index'],
    mode='lines',
    name='Insured Losses',
    line=dict(color=COLORS['positive'], width=2.5),
    yaxis='y1'
))

fig_b.add_trace(go.Scatter(
    x=df_gap['year'],
    y=df_gap['emissions_index'],
    mode='lines',
    name='GHG Emissions',
    line=dict(color=COLORS['negative'], width=2.5),
    yaxis='y2'
))

fig_b.add_annotation(
    x=0.01, y=1.18,
    xref='paper', yref='paper',
    text='Both series indexed to 1991 = 100. Nominal losses (GDV). Emissions: UBA KSG sectors.<br>Note: 1991 was an unusually quiet loss year (€1.1bn) — one of the lowest on record.<br>Indexing from a low base makes the losses line appear more volatile than from a typical year.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig_b.update_layout(
    title='Germany — GHG Emissions vs Insured Losses: Diverging Trends (1991–2024)',
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        title='Insured Losses Index (1991 = 100)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['positive'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['positive'],
    ),
    yaxis2=dict(
        title='GHG Emissions Index (1991 = 100)',
        overlaying='y',
        side='right',
        showgrid=False,
        showline=True,
        linecolor=COLORS['negative'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['negative'],
    ),
    height=480,
    margin=dict(t=120),
)

fig_b.show()
fig_b.write_html(FIGURES / "03_adaptation_gap_index.html")
print("Chart B saved ✓")

Chart B saved ✓


### 📝 How to read this chart — plain language explanation

**What is an index chart?**
An index chart converts two series measured in different units into a common scale,
so they can be compared visually. Here, losses are in € billions and emissions are
in Mt CO₂eq — you can't plot those on the same axis meaningfully. By setting both
to 100 at the starting point (1991), we ask a simpler question:
*relative to where each started, how much has each changed?*

**How the index is calculated:**
Each series is divided by its own 1991 value and multiplied by 100 — independently:
- `losses_index = (losses / losses_1991) × 100`
- `emissions_index = (emissions / emissions_1991) × 100`

So the 100 baseline means something different for each series — €1.1bn for losses,
1206 Mt for emissions. After that, you're no longer comparing euros to tonnes.
You're comparing *how much each has grown or shrunk relative to its own starting point*.

**What the chart shows:**
- The emissions line (violet) trends steadily downward toward ~50 by 2024 —
  Germany has roughly halved its GHG emissions since 1991. Decarbonisation is real.
- The losses line (orange) spikes dramatically — reaching 400–1400 in bad years —
  and shows no clear downward trend despite falling emissions.

**Why this matters for RQ4:**
This is the adaptation gap in one picture. Germany is successfully reducing the
*cause* of climate change (emissions), but the *consequences* (insured losses from
extreme weather) are not following the same downward path. Damage locked in from
decades of past emissions continues to materialise — regardless of what we do today.

**One caveat:**
1991 was an unusually quiet loss year (€1.1bn — one of the lowest on record).
Indexing from a low base makes the orange line look more dramatic. A different
base year would change the visual impression but not the underlying finding.

In [47]:
# --- Chart A: Absolute losses vs emissions, dual axis (1991–2024) ---

df_a = pd.merge(
    df_merged[['year', 'total_damage_mrd']],
    df_uba[['year', 'total_emissions_mio_t']],
    on='year',
    how='inner'
)

fig_a = go.Figure()

fig_a.add_trace(go.Bar(
    x=df_a['year'],
    y=df_a['total_damage_mrd'],
    name='Insured Losses (€ bn)',
    marker_color=COLORS['positive'],
    marker_line_width=0,
    opacity=0.85,
    yaxis='y1'
))

fig_a.add_trace(go.Scatter(
    x=df_a['year'],
    y=df_a['total_emissions_mio_t'],
    mode='lines',
    name='GHG Emissions (Mt CO₂eq)',
    line=dict(color=COLORS['negative'], width=2.5),
    yaxis='y2'
))

fig_a.add_annotation(
    x=0.01, y=1.18,
    xref='paper', yref='paper',
    text='Nominal insured losses (GDV, left axis). Total GHG emissions UBA KSG sectors (right axis).',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig_a.update_layout(
    title='Germany — Insured Losses vs GHG Emissions (1991–2024)',
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
    ),
    yaxis=dict(
        title='Insured Losses (€ billion)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['positive'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['positive'],
    ),
    yaxis2=dict(
        title='GHG Emissions (Mt CO₂eq)',
        overlaying='y',
        side='right',
        showgrid=False,
        showline=True,
        linecolor=COLORS['negative'],
        linewidth=1.2,
        zeroline=False,
        color=COLORS['negative'],
    ),
    height=480,
    margin=dict(t=120),
)

fig_a.show()
fig_a.write_html(FIGURES / "03_losses_vs_emissions_dual_axis.html")
print("Chart A saved ✓")

Chart A saved ✓


In [56]:
# ============================================================
# SECTION 4c — KSG PATHWAY CHART (inspired by Climate Action Tracker)
# ============================================================

# --- Historical emissions (UBA 1990–2025) ---
df_historical = df_uba[['year', 'total_emissions_mio_t']].copy()

# --- KSG legally binding targets ---
ksg_years  = [2025, 2030, 2040, 2045]
ksg_values = [656,   438,  219,    0]

# --- Current policies projection (CAT) ---
cap_years  = [2025, 2030, 2035, 2040, 2045]
cap_values = [656,   600,  560,  520,  480]

# --- 1.5°C compatible pathway (CAT fair share) ---
path_years  = [2025, 2030, 2035, 2040, 2045]
path_values = [600,   320,  160,   80,    0]

# Connect historical to projections at 2025
last_historical_year  = df_historical['year'].max()
last_historical_value = df_historical.loc[
    df_historical['year'] == last_historical_year, 'total_emissions_mio_t'
].values[0]

print(f"Last historical year:  {last_historical_year}")
print(f"Last historical value: {last_historical_value} Mt")
print("Data ready ✓")

Last historical year:  2025
Last historical value: 648.87 Mt
Data ready ✓


In [57]:
# --- Update projection start to match actual 2025 value ---
ksg_years  = [2025, 2030, 2040, 2045]
ksg_values = [648.87, 438, 219, 0]

cap_years  = [2025, 2030, 2035, 2040, 2045]
cap_values = [648.87, 600, 560, 520, 480]

path_years  = [2025, 2030, 2035, 2040, 2045]
path_values = [648.87, 320, 160, 80, 0]

# --- Build chart ---
fig_ksg = go.Figure()

# Historical line
fig_ksg.add_trace(go.Scatter(
    x=df_historical['year'],
    y=df_historical['total_emissions_mio_t'],
    mode='lines',
    name='Historical (UBA)',
    line=dict(color=COLORS['accent'], width=2.5),
))

# KSG target pathway
fig_ksg.add_trace(go.Scatter(
    x=ksg_years,
    y=ksg_values,
    mode='lines+markers',
    name='KSG targets (legally binding)',
    line=dict(color=COLORS['positive'], width=2, dash='dash'),
    marker=dict(size=7, color=COLORS['positive']),
))

# Current policies projection
fig_ksg.add_trace(go.Scatter(
    x=cap_years,
    y=cap_values,
    mode='lines+markers',
    name='Current policies (CAT projection)',
    line=dict(color='#E8A838', width=2, dash='dot'),
    marker=dict(size=7, color='#E8A838'),
))

# 1.5°C compatible pathway
fig_ksg.add_trace(go.Scatter(
    x=path_years,
    y=path_values,
    mode='lines+markers',
    name='1.5°C compatible pathway (CAT)',
    line=dict(color=COLORS['negative'], width=2, dash='dashdot'),
    marker=dict(size=7, color=COLORS['negative']),
))

# Shade gap between current policies and 1.5°C pathway
fig_ksg.add_trace(go.Scatter(
    x=cap_years + path_years[::-1],
    y=cap_values + path_values[::-1],
    fill='toself',
    fillcolor='rgba(232,232,232,0.06)',
    line=dict(color='rgba(0,0,0,0)'),
    name='Policy gap',
    showlegend=True,
    hoverinfo='skip',
))

# Vertical line at 2025 separating historical from projections
fig_ksg.add_vline(
    x=2025,
    line_color=COLORS['accent'],
    line_width=1,
    line_dash='dash',
    opacity=0.4,
)

fig_ksg.add_annotation(
    x=2025, y=1100,
    text='← Historical | Projections →',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    xanchor='center',
)

fig_ksg.add_annotation(
    x=0.01, y=1,
    xref='paper', yref='paper',
    text='Historical: UBA KSG sectors. KSG targets: Klimaschutzgesetz (German Climate Protection Act).<br>CAT projections: Climate Action Tracker, July 2025 update. Values in Mt CO₂eq.',
    showarrow=False,
    font=dict(color=COLORS['text_muted'], size=10),
    align='left'
)

fig_ksg.update_layout(
    title="Germany — GHG Emissions: Historical Trajectory & Future Pathways (1990–2045)",
    title_font=dict(color=COLORS['text']),
    plot_bgcolor=COLORS['background'],
    paper_bgcolor=COLORS['background'],
    legend=dict(
        font=dict(color=COLORS['text']),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        color=COLORS['text_muted'],
        dtick=5,
        range=[1990, 2046],
    ),
    yaxis=dict(
        title='GHG Emissions (Mt CO₂eq)',
        showgrid=True,
        gridcolor=COLORS['grid'],
        showline=True,
        linecolor=COLORS['accent'],
        linewidth=1.2,
        zeroline=True,
        zerolinecolor=COLORS['accent'],
        zerolinewidth=0.8,
        color=COLORS['text_muted'],
        range=[-50, 1350],
    ),
    height=520,
    margin=dict(t=120),
)

fig_ksg.show()
fig_ksg.write_html(FIGURES / "03_ksg_pathway.html")
print("KSG pathway chart saved ✓")

KSG pathway chart saved ✓


In [58]:
# ============================================================
# SAVE PROCESSED DATA FOR NOTEBOOK 04
# ============================================================

# Save merged dataframe (GDV + GVA + CPI)
df_merged.to_csv(PROCESSED / "gdv_gva_merged_1991_2024.csv", index=False)

# Save GDV with real losses (full 1973–2024 series)
df_gdv_real.to_csv(PROCESSED / "gdv_damage_real_1973_2024.csv", index=False)

print("gdv_gva_merged_1991_2024.csv saved ✓")
print("gdv_damage_real_1973_2024.csv saved ✓")

gdv_gva_merged_1991_2024.csv saved ✓
gdv_damage_real_1973_2024.csv saved ✓
